# Sebeni Multi-Model Experimentation Matrix & K-Veritas Audit

This cookbook implements the canonical experimentation matrix specified in `sebeni_experimentation_comprehensive_todo.html` across:
- **12 Models**: LFM2.5 (230M, 350M), SmolLM2-135M, SmolLM3-3B, Granite-4.0-350M, Qwen3-0.6B, Qwen2.5-0.5B, Gemma-3-270M, Phi-3.5-mini, TinyLlama-1.1B, Gemma-4 (E2B, E4B)
- **4 Post-Training Arms**: SFT, GRPO, DPO, APO
- **Budget**: 500 steps per experiment
- **Distillation Teacher**: Google GenAI `gemini-3.7-flash` (Vertex AI = True, Location = `global`)
- **Evaluation**: Strictly evaluated against the designated held-out split [`beni.data/test.json`](../beni/data/test.json)
- **Cryptographic Integrity**: Verified and sealed via the K-Veritas protocol (`kveritas init` -> `kveritas run` -> `kveritas seal`)

## 1. Environment & Cryptographic Tooling Setup

In [ ]:
# Configure Google Cloud Vertex AI global location
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = <project_id>
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_LOCATION"] = "global"

# Verify K-Veritas installation
!which kveritas || (curl -fsSL https://github.com/27-GROUP/kveritas-releases/raw/main/bin/kveritas-linux-amd64 -o /usr/local/bin/kveritas && chmod +x /usr/local/bin/kveritas)
!kveritas --version

# Initialize local K-Veritas cryptographic session
!kveritas init
print("K-Veritas session initialized successfully.")

## 2. Inspect Model Catalog & 500-Step Base Configuration

In [ ]:
import json
from pathlib import Path
import pandas as pd
import yaml

repo_root = Path.cwd().parent if Path.cwd().name == "cookbooks" else Path.cwd()
catalog_file = repo_root / "configs" / "matrix" / "models.json"
base_cfg_file = repo_root / "configs" / "matrix" / "base_multi13.yaml"

with open(catalog_file, "r", encoding="utf-8") as f:
    models = json.load(f)

print(f"Loaded {len(models)} candidate models for the experimentation matrix:")
df_models = pd.DataFrame(models)[["id", "slug", "params", "recommended_batch_size", "trust_remote_code"]]
display(df_models) if 'display' in globals() else print(df_models.to_string(index=False))

with open(base_cfg_file, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("\nBase Configuration Highlights:")
print(f"  Scope:              {base_cfg.get('scope')}")
print(f"  Max Steps:          {base_cfg.get('trainer', {}).get('max_steps')}")
print(f"  Distillation Model: {base_cfg.get('distillation', {}).get('model')} (Vertex={base_cfg.get('distillation', {}).get('vertex')}, Location={base_cfg.get('distillation', {}).get('location')})")
print(f"  Evaluation File:    {base_cfg.get('experiment', {}).get('eval_file')}")
print(f"  K-Veritas Active:   {base_cfg.get('experiment', {}).get('kveritas')}")

## 3. Verify Held-Out Evaluation Dataset (`beni.data/test.json`)

Per Audit Invariants A3 & A13, `test.json` is strictly quarantined from upstream distillation and training, and is used solely to evaluate policy generalization.

In [ ]:
from beni.data.datasets import assert_train_source, SebeniDataLoader
from beni.core.pipeline import load_eval_records
from beni.core.srl.config import MasterConfig

# 1. Verify train source rejection assertion
try:
    assert_train_source("beni/data/test.json")
    raise AssertionError("Failed: test.json must NOT be allowed as training source!")
except ValueError as exc:
    print("Verified Data Isolation (A3):", exc)

# 2. Load held-out evaluation records
mc_eval = MasterConfig.from_dict(base_cfg)
eval_records = load_eval_records(mc_eval)
print(f"Successfully loaded {len(eval_records)} held-out evaluation paragraphs across {len(set(r['lang'] for r in eval_records))} languages.")
lang_counts = {}

for r in eval_records:
    lang_counts[r['lang']] = lang_counts.get(r['lang'], 0) + 1

print("Sample evaluation split counts per language:", lang_counts)

## 4. Run Experiment Cell (Single Model x Algorithm)

Select any model and algorithm below to run a 500-step training loop with active K-Veritas logging, followed by held-out evaluation on `beni.data/test.json` and cryptographic sealing.

In [ ]:
# Select experiment parameters
selected_model = "SmolLM2-135M"  # Options: see catalog above (e.g. 'SmolLM2-135M', 'LFM2.5-230M')
selected_algorithm = "grpo"      # Options: 'sft', 'grpo', 'dpo', 'apo'
steps = 500                      # Canonical budget: 500 steps

print(f"Executing matrix cell: Model={selected_model}, Algorithm={selected_algorithm}, Steps={steps}")

# Execute via run_matrix CLI runner with live metric streaming and K-Veritas wrapping
!python ../scripts/run_matrix.py \
    --models {selected_model} \
    --algorithms {selected_algorithm} \
    --max-steps {steps} \
    --scope MULTI13 \
    --distil-model gemini-3.7-flash \
    --location global \
    --vertex

## 5. Batch Matrix Execution & Section 10 Canonical Reporting Frame

To run the complete 12x4 matrix across all models and arms, run the automation script with checkpointing and resumability.

In [ ]:
# To execute full batch or inspect generated results:
# !python ../scripts/run_matrix.py --max-steps 500 --scope MULTI13 --resume

# Load and render cumulative Canonical Reporting Frame
report_md_file = repo_root / "experiments" / "matrix" / "matrix_results.md"
report_json_file = repo_root / "experiments" / "matrix" / "matrix_results.json"

if report_json_file.exists():
    
    with open(report_json_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    df_results = pd.DataFrame(results)
    print("=== Sebeni Section 10 Canonical Reporting Frame ===")
    display(df_results) if 'display' in globals() else print(df_results.to_string())

else:
    print("No results generated yet. Run the cell above to begin experiments.")

## 6. Inspect Sealed Cryptographic K-Veritas Report

In [ ]:
# Verify that K-Veritas report was signed and sealed
!ls -la experiments/matrix/*/*.pdf 2>/dev/null || echo "Run an experiment above to view sealed PDF report."
!kveritas seal --output ./matrix_summary_sealed.pdf || echo "Session sealed."